In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
import importlib
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore', message='Mean of empty slice')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000016', 'SEQ_000018',
                    'SEQ_000022', 'SEQ_000033', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053',
                    'SEQ_000058', 'SEQ_000063', 'SEQ_000079', 'SEQ_000091', 'SEQ_000092',
                    'SEQ_000111', 'SEQ_000113', 'SEQ_000114', 'SEQ_000142', 'SEQ_000150']
some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165',
                'SUBJ_024086', 'SUBJ_040733', 'SUBJ_063346', 'SUBJ_055211', 'SUBJ_001430',
                'SUBJ_012088', 'SUBJ_040310', 'SUBJ_032233', 'SUBJ_059330', 'SUBJ_013623',
                'SUBJ_032585', 'SUBJ_063464', 'SUBJ_038023', 'SUBJ_044680', 'SUBJ_024137']

model_run_name = 'xgboost.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

train_sample_pct = 0.20
test_sample_pct = 0.05

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

In [ ]:
train_df = raw_train_df.set_index('row_id')

some_subjects_df = train_df[train_df['subject'].isin(some_subjects)]

multiple_gesture_df = train_df[train_df['sequence_id'].isin(['SEQ_065471', 'SEQ_065478', 'SEQ_065487'])]

train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.2, test_pct=0.2)

In [ ]:
# 0.47 across all domains

importlib.reload(utils)

num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = multiple_gesture_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list': [[None], ['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['time', 'velocity', 'frequency', 'acceleration'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [[None], ['rot_w','rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes': [True],
    f'{pipe_name}__rotation_domain': ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [[None], ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],

    f'{pipe_name}__tof_sensor_list': [tof_columns],
    f'{pipe_name}__tof_mode': ['simple', 'research'],

    f'{pipe_name}__window': [0.25],
    f'{pipe_name}__step_sec': [0.1],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],

    f'{pipe_name}__category_data': [True],
    
    f'{pipe_name}__segmentation': [None, 'window' 'phase'],

    'classifier__estimator__n_estimators': [200],
    'classifier__estimator__max_depth': [4],
    'classifier__estimator__learning_rate': [0.05],
    'classifier__estimator__subsample': [0.8],
    'classifier__estimator__colsample_bytree': [0.8],
    'classifier__estimator__min_child_weight': [5],
    'classifier__estimator__gamma': [0],
    'classifier__estimator__reg_alpha': [0],
    'classifier__estimator__reg_lambda': [1],
}

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z']
)

n_classes = sliced_data_df['gesture'].nunique()

xgb_clf = XGBClassifier(
    objective='multi:softmax',
    num_class=n_classes,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator = xgb_clf, 
        extractor = custom_extractor, 
        mode = 'xgboost'
        ))
])

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', 'gesture']]
n_fits = len(list(ParameterGrid(param_grid))) * n_splits

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])
pbar.close()

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)

In [ ]:
cv_results_df.T

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']

model_run_name = 'xgboost.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

train_sample_pct = 0.20
test_sample_pct = 0.05

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

In [ ]:
train_df = raw_train_df.set_index('row_id')